# Thyroid cancer vs. normal from miRNA expression with Neural Additive Models

M.Sc. thesis project, Sharif University of Technology (2024).

Pipeline: TCGA-THCA miRNA counts → ACTGAN synthetic augmentation → PyDESeq2 normalization and filtering → feature selection (t-score / information gain / chi² / ReliefF) → NAM classifier → per-miRNA shape functions.

Written for Google Colab. The data are not included in the repo (see the README).

In [ ]:
!pip install -q "pydeseq2>=0.5" skrebate tf-keras
!pip install -q -e "git+https://github.com/agarwl/google-research.git#egg=neural_additive_models&subdirectory=neural_additive_models"

Restart the runtime after the install cell, then run from here.

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'  # the NAM code is written for Keras 2

import glob
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import tensorflow.compat.v1 as tf

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from neural_additive_models import data_utils, graph_builder

sys.path.append('../src')
from feature_selection import select_features

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/My Drive/thyroid_mirna/'
else:
    DATA_DIR = os.environ.get('DATA_DIR', '../data/')

SEED = 0
rng = np.random.default_rng(SEED)

## 1. Real TCGA-THCA samples

GDC miRNA quantification files (`*.mirbase21.mirnas.quantification.txt`), 50 primary tumor and 50 solid tissue normal samples. These 100 samples were used to train the ACTGAN generators.

In [ ]:
def read_gdc_folder(folder):
    counts = {}
    for f in sorted(glob.glob(os.path.join(folder, '*.txt'))):
        name = os.path.basename(f).split('.')[0]
        df = pd.read_csv(f, sep='\t')
        counts[name] = df.set_index('miRNA_ID')['read_count']
    return pd.DataFrame(counts).T

real_cancer = read_gdc_folder(DATA_DIR + 'gdc/Cancer/')
real_normal = read_gdc_folder(DATA_DIR + 'gdc/Normal/')
mirna_ids = real_cancer.columns

print(real_cancer.shape, real_normal.shape)

## 2. Synthetic samples

ACTGAN was trained separately on each class (cancer / normal). Here I read 5,000 generated samples per class.

In [ ]:
N_SYN = 5000

syn_cancer = pd.read_csv(DATA_DIR + 'synthetic/actgan-cancer-data.csv', index_col=0).iloc[:N_SYN]
syn_normal = pd.read_csv(DATA_DIR + 'synthetic/actgan-normal-data.csv', index_col=0).iloc[:N_SYN]

counts = pd.concat([syn_cancer, syn_normal]).clip(lower=0).round().astype(int)
counts.columns = mirna_ids
counts.index = [f'cancer_{i}' for i in range(len(syn_cancer))] + [f'normal_{i}' for i in range(len(syn_normal))]

condition = ['Cancer'] * len(syn_cancer) + ['Normal'] * len(syn_normal)
metadata = pd.DataFrame({'condition': condition}, index=counts.index)
y = (metadata['condition'] == 'Normal').astype(int).to_numpy()

counts.shape

## 3. DESeq2 normalization and filtering

In [ ]:
dds = DeseqDataSet(
    counts=counts,
    metadata=metadata,
    design='~condition',
    refit_cooks=True,
    n_cpus=8,
    quiet=True,
)
dds.deseq2()

In [ ]:
stat_res = DeseqStats(dds, contrast=['condition', 'Cancer', 'Normal'], quiet=True)
stat_res.summary()
stat_res.lfc_shrink(coeff='condition[T.Normal]')

res = stat_res.results_df
de_mirnas = res.index[res['pvalue'] <= 1e-5]
print(len(de_mirnas), 'of', res.shape[0], 'miRNAs with p <= 1e-5')

In [ ]:
norm = pd.DataFrame(dds.layers['normed_counts'], index=counts.index, columns=counts.columns)
norm = norm[de_mirnas]
norm.shape

## 4. Feature selection

Each method was run in three rounds:
1. top-k miRNAs from the selection method,
2. keep only the miRNAs whose NAM shape function is not flat (read off the plots in section 6),
3. add random miRNAs from the filtered set, as a check that the selected ones still carry the prediction.

In [ ]:
METHOD = 't_score'   # 't_score', 'info_gain', 'chi2', 'relieff'
K = 20

selected = select_features(norm, y, METHOD, K)
selected

In [ ]:
# round 2
keep = None
# keep = ['hsa-mir-150', 'hsa-mir-93', 'hsa-mir-199b', 'hsa-mir-191', 'hsa-mir-195']   # t-score
# keep = ['hsa-mir-221', 'hsa-mir-10a', 'hsa-mir-181b', 'hsa-mir-150']                  # information gain
if keep is not None:
    selected = [m for m in selected if m in keep]

# round 3
N_RANDOM = 0
if N_RANDOM > 0:
    pool = [m for m in norm.columns if m not in selected]
    selected = selected + list(rng.choice(pool, N_RANDOM, replace=False))

X_df = norm[selected]
print(len(selected), 'features')

In [ ]:
corr = X_df.corr()
plt.figure(figsize=(0.5 * len(selected) + 4, 0.4 * len(selected) + 3))
sns.heatmap(corr, vmin=-1, vmax=1, cmap='vlag', square=True, annot=len(selected) <= 10, fmt='.2f')
plt.title(f'{METHOD}: correlation of selected miRNAs')
plt.show()

## 5. NAM

Google's NAM implementation (`neural_additive_models`, Agarwal et al., 2021). Inputs are min-max scaled to [-1, 1] per feature, the same as in their data loaders. Folds and the train/validation split use their helpers; the hyperparameters are picked on the validation split and the test fold is only used at the end.

In [ ]:
col_min_max = {c: (X_df[c].min(), X_df[c].max()) for c in X_df.columns}

def min_max_scale(x, min_val, max_val):
    return 2 * (x - min_val) / (max_val - min_val) - 1

def inverse_min_max_scale(x, min_val, max_val):
    return (x + 1) / 2 * (max_val - min_val) + min_val

data_x = np.stack([min_max_scale(X_df[c].to_numpy(), *col_min_max[c]) for c in X_df.columns], axis=1).astype(np.float32)
data_y = y.astype(np.float32)
column_names = list(X_df.columns)

(x_train_all, y_train_all), (x_test, y_test) = data_utils.get_train_test_fold(
    data_x, data_y, fold_num=1, num_folds=5, stratified=True)
data_gen = data_utils.split_training_dataset(x_train_all, y_train_all, n_splits=20, stratified=True)
(x_train, y_train), (x_val, y_val) = next(data_gen)

print(x_train.shape, x_val.shape, x_test.shape)

In [ ]:
def predict(nn_model, x, batch_size=1024):
    out = [nn_model(x[i:i + batch_size], training=False).numpy() for i in range(0, len(x), batch_size)]
    return np.concatenate(out)

def accuracy(nn_model, x, y):
    return np.mean((predict(nn_model, x) > 0) == (y > 0.5))

def train_nam(dropout, num_basis_functions, epochs=30, lr=1e-2, batch_size=1024,
              output_regularization=0.0, l2_regularization=1e-6):
    tf.keras.backend.clear_session()
    nn_model = graph_builder.create_nam_model(
        x_train=x_train,
        dropout=dropout,
        num_basis_functions=num_basis_functions,
        activation='relu',
        trainable=True,
        shallow=False,
        name_scope='model_0')
    optimizer = tf.keras.optimizers.Adam(lr)

    for epoch in range(epochs):
        idx = rng.permutation(len(x_train))
        for i in range(0, len(idx), batch_size):
            b = idx[i:i + batch_size]
            with tf.GradientTape() as tape:
                loss = graph_builder.penalized_cross_entropy_loss(
                    nn_model, x_train[b], y_train[b], output_regularization, l2_regularization)
            grads = tape.gradient(loss, nn_model.trainable_variables)
            optimizer.apply_gradients(zip(grads, nn_model.trainable_variables))
    return nn_model

In [ ]:
dropouts = [0, 0.05, 0.1, 0.15]
n_basis = [2, 3, 4, 5, 6, 7, 8, 16, 32, 64]

results = []
best_acc, best_model = -1, None
for d in dropouts:
    for nb in n_basis:
        nn_model = train_nam(d, nb)
        val_acc = accuracy(nn_model, x_val, y_val)
        results.append((d, nb, val_acc))
        print(f'dropout={d:<5} basis={nb:<3} val_acc={val_acc:.4f}')
        if val_acc > best_acc:
            best_acc, best_model, best_params = val_acc, nn_model, (d, nb)

results = pd.DataFrame(results, columns=['dropout', 'num_basis_functions', 'val_acc'])

In [ ]:
test_pred = predict(best_model, x_test)
test_acc = np.mean((test_pred > 0) == (y_test > 0.5))
test_auc = graph_builder.calculate_metric(y_test, test_pred, regression=False)

print('best (dropout, basis):', best_params)
print(f'{METHOD}, {len(selected)} miRNAs: test accuracy = {100 * test_acc:.2f}%, AUROC = {test_auc:.4f}')

## 6. Shape functions

Each miRNA gets its own small network f_i, and the model output is logit = bias + Σ f_i(x_i), so plotting f_i against the (unscaled) normalized count shows exactly how that miRNA pushes the prediction. Red shading is the data density. Positive values push toward **normal**, negative toward **cancer**.

In [ ]:
unique_x = [np.unique(data_x[:, i]) for i in range(data_x.shape[1])]
feature_pred = {}
mean_pred = {}
for i, name in enumerate(column_names):
    f = best_model.feature_nns[i](unique_x[i][:, None], training=best_model._false).numpy()
    feature_pred[name] = f
    pos = np.searchsorted(unique_x[i], data_x[:, i])
    mean_pred[name] = f[pos].mean()

importance = {n: np.mean(np.abs(feature_pred[n][np.searchsorted(unique_x[i], data_x[:, i])] - mean_pred[n]))
              for i, n in enumerate(column_names)}
importance = pd.Series(importance).sort_values()

plt.figure(figsize=(6, 0.3 * len(importance) + 1.5))
importance.plot.barh(color='#5b7bd5')
plt.xlabel('mean |f_i(x_i)|')
plt.title(f'Global importance ({METHOD})')
plt.tight_layout()
plt.show()

In [ ]:
def plot_shape_functions(names, num_cols=5, n_blocks=100, color=[0.4, 0.5, 0.9]):
    num_rows = int(np.ceil(len(names) / num_cols))
    fig = plt.figure(figsize=(num_cols * 4.5, num_rows * 4))

    curves = {n: feature_pred[n] - mean_pred[n] for n in names}
    min_y = min(c.min() for c in curves.values())
    max_y = max(c.max() for c in curves.values())
    pad = 0.05 * (max_y - min_y + 1e-9)
    min_y, max_y = min_y - pad, max_y + pad

    for k, name in enumerate(names):
        i = column_names.index(name)
        x_orig = inverse_min_max_scale(unique_x[i], *col_min_max[name])
        all_orig = X_df[name].to_numpy()

        ax = plt.subplot(num_rows, num_cols, k + 1)
        density, edges = np.histogram(all_orig, bins=min(n_blocks, len(x_orig)))
        density = density / density.max()
        for p in range(len(density)):
            ax.add_patch(patches.Rectangle((edges[p], min_y), edges[p + 1] - edges[p], max_y - min_y,
                                           linewidth=0, facecolor=[0.9, 0.5, 0.5],
                                           alpha=min(1.0, 0.01 + density[p])))
        ax.plot(x_orig, curves[name], color=color, linewidth=3)
        ax.set_xlim(x_orig.min(), x_orig.max())
        ax.set_ylim(min_y, max_y)
        ax.set_xlabel(name, fontsize='x-large')
        if k % num_cols == 0:
            ax.set_ylabel('contribution to logit', fontsize='x-large')

    plt.subplots_adjust(hspace=0.3, wspace=0.3)
    return fig

fig = plot_shape_functions(column_names)
plt.show()

In [ ]:
os.makedirs('../figures', exist_ok=True)
fig.savefig(f'../figures/shape_{METHOD}_{len(selected)}.png', dpi=150, bbox_inches='tight')